# Hermes Agent 能力篇：会话、工具、MCP 与持久记忆


## 模型接通之后，Agent 怎样持续工作？

第 01 篇完成安装、模型接入和配置定位。本章开始理解 Agent 的工作环境：如何恢复任务，如何调用工具，如何接入外部服务，以及哪些信息能跨会话保留。

| 本章模块 | 要解决的问题 |
| --- | --- |
| 会话管理 | 怎样继续任务、控制上下文、搜索过去的讨论？ |
| Dashboard | 怎样用浏览器查看配置、凭证和会话状态？ |
| Toolsets | 怎样选择可用工具、观察执行，并确定执行环境？ |
| MCP | 怎样把外部工具接入 Hermes，或让其他客户端连接 Hermes？ |
| 上下文文件 | 项目规则、实例风格怎样进入模型上下文？ |
| 持久记忆 | 哪些稳定事实应当保存，何时在下一次会话生效？ |

本篇依据《02-能力篇.md》整理，课件以 v0.16.0 为基准。工具字段和部分实现另对照了当前公开源码，遇到变化会标明，不能把当前示例当作所有版本通用接口。

所有单元格均为 Markdown。命令、配置和工具调用参数只供阅读，没有实际启动 Dashboard、MCP、容器或 Agent 工具。


## 会话：当前任务的工作上下文

一个 Session 保存任务的消息、工具调用和相关状态。继续同一会话，可以沿用已有讨论；开启新会话，则重新组织本次上下文，并按配置载入项目规则与记忆。

~~~text
当前会话：用户消息 → 模型决策 → 工具结果 → 后续消息
历史存储：记录会话与消息，供恢复和搜索
新会话：新任务上下文 + 项目说明 + 持久记忆快照
~~~

历史已保存，不等于模型每次都看到全部历史。上下文窗口有限，压缩、搜索和记忆分别解决不同问题。

任务主题明显变化时，可以建立新会话；需要延续未完成任务时，先恢复对应会话。仅重启进程不一定等于开启新会话，尤其是消息平台中的持续对话。


## 列出、浏览与继续会话

~~~bash
hermes sessions list
hermes sessions browse
hermes --continue
hermes -c
~~~

| 命令 | 用途 |
| --- | --- |
| sessions list | 查看近期会话，找到标题和 ID |
| sessions browse | 打开交互式选择器 |
| --continue / -c | 继续一个最近会话，二者为长短参数形式 |

上面列出的是不同入口，不需要按顺序连续启动多个交互会话。

课件把 -c 概括为“继续上次会话”。当前版本还可能依据终端、工作目录与实例选择续聊对象，多终端并行时应检查实际恢复了哪一个会话；精确指定任务时优先使用会话 ID。


## 按 ID 或标题恢复，再给任务起清楚的名字

~~~bash
hermes --resume 20260610_143052_a1b2c3
hermes -r "我的项目设置"
hermes sessions rename 20260610_143052_a1b2c3 "后端 API 开发"
~~~

日期形式的 ID 和标题都是示例，需要替换成 sessions list 中实际存在的值。重命名改变会话的可识别标题，不是重写所有历史消息。

可以用“项目 + 任务”命名，例如“博客系统：Docker 端口排查”，比“问题解决”更容易在恢复和搜索时区分。

标题可能重复，来源平台也可能不同。恢复后先查看当前目标、最后结论和执行环境，避免把另一个会话中的决定套到当前项目。


## 会话内命令：新建、查看、撤销与后台工作

以下命令在 Hermes 会话输入框中使用，不是在 PowerShell 或 Bash 中执行。

~~~text
/new
/clear
/title 后端 API 开发
/history
/sessions
/undo
/undo 2
/compress
~~~

| 命令 | 课件中的主要用途 | 需要分清 |
| --- | --- | --- |
| /new | 建立新会话 | 不是删除全部已保存历史 |
| /clear | 清屏并开始新会话 | 清屏不等于清除数据库中的所有记录 |
| /title | 给当前会话命名 | 修改标题，与模型或目录配置不同 |
| /history / /sessions | 查看历史或管理会话 | 可见范围依入口和实例而定 |
| /undo [N] | 撤销最近一轮或多轮交互 | 不应据此假定工具的外部效果也已回滚 |
| /compress | 手动压缩当前上下文 | 压缩不是历史删除或完整备份 |

课件还介绍 /background、/stop 和 /goal，分别涉及后台任务、停止请求和持续目标。停止作用依当前入口和任务实现而定；它不自动撤销已经发生的文件修改、部署或外部请求。

### 撤销聊天后，还要检查实际结果

文件是否恢复，应查看文件内容、Git diff 或对应检查点；数据库写入和消息发送等外部效果要通过各自系统确认。聊天记录中没有了某一步，不能作为那一步从未执行的证明。


## 删除与清理会话：先确定对象和保留需求

~~~bash
hermes sessions delete --help
hermes sessions prune --help
hermes sessions delete 20260610_143052_a1b2c3
hermes sessions prune --older-than 30
~~~

delete 指向一个具体会话；prune 按版本支持的规则清理旧会话。30 表示课件示例中的天数，不是要清理的会话数量。

先查看所用版本的帮助，确认结束状态、归档、固定会话及确认流程等条件。需要保留的讨论应先按对应版本导出或备份。

当前公开实现指出：删除仍在运行的会话，并不会停止它；后续保存可能重新建立同一个 ID 的记录。需要删除时先结束相应活动，再执行所需的数据清理。这里没有实际删除或清理会话。


## 会话主要存在 SQLite 中，目录树只是部分视图

课件以数据根目录下的 state.db 作为会话查询、恢复和搜索的主要存储。默认位置延续上一章：

| 环境 | 默认会话数据库 |
| --- | --- |
| Linux / macOS / WSL | `~/.hermes/state.db` |
| Windows 原生 | `%LOCALAPPDATA%\hermes\state.db` |
| 自定义 HERMES_HOME / Profile | 对应实例实际数据根目录中的数据库 |

| 表 / 索引 | 主要内容 |
| --- | --- |
| sessions | 会话 ID、标题、来源、模型及其他元数据 |
| messages | 消息正文、角色、工具调用与时间等 |
| state_meta / schema_version | 状态元数据与数据库迁移版本 |
| messages_fts | 全文检索索引 |
| messages_fts_trigram | CJK 子串检索相关索引 |

FTS5 还维护自己的影子表，它们是索引实现的一部分，不宜把每张表当作业务数据手工改写。

SQLite 的 WAL 模式允许读者与写入更好地并发，但仍是单写者机制。运行中的数据库可能还有 -wal、-shm 文件；备份要使用支持一致性的方式，不能只复制 state.db 就保证得到完整状态。

课件提到早期逐会话 JSONL 转录。当前存储布局会随版本增加其他文件，因此只备份 sessions 目录也不等于备份全部聊天历史。


## 上下文压缩：减少当前请求中的历史负担

长会话会累计用户消息、工具输出和模型回复。接近上下文限制时，Hermes 可以压缩较早内容，保留摘要和近期工作信息。

课件配置如下，合并到数据目录的 config.yaml 对应节点中：

~~~yaml
compression:
  enabled: true
  threshold: 0.50
  target_ratio: 0.20
  protect_last_n: 20
~~~

| 参数 | 课件中的含义 |
| --- | --- |
| enabled | 是否启用压缩 |
| threshold | 触发阈值比例，例如约到上下文窗口的 50% |
| target_ratio | 近期消息保留预算，按触发阈值预算的一部分计算 |
| protect_last_n | 保护近期消息的数量要求 |

例如窗口是 100000 tokens，threshold=0.50 对应 50000 的触发量级；target_ratio=0.20 对应约 10000 的近期保留预算。**不是压缩后全部提示词一定只有 10000 tokens。**系统提示、摘要、工具定义和其他保护内容仍占预算。

保护的是消息，不一定等于 20 轮用户问答。模型差异、输出预留、超大工具结果及实现版本会影响实际预算，不能把四个参数解释为精确的最终上下文长度保证。


## 压缩、删除、记忆与搜索，各做一件事

| 动作 | 改变什么 | 不等同于什么 |
| --- | --- | --- |
| /compress | 当前模型上下文的组织与长度 | 删除历史记录 |
| /new | 当前任务的会话边界 | 忘掉所有持久记忆 |
| sessions delete / prune | 所选历史存储记录 | 回滚此前工具行为 |
| memory | 保存少量长期事实 | 保存整份聊天转录 |
| session_search | 按需找历史证据 | 读取当前文件或远端系统的实时状态 |

摘要可能遗漏条件或细节，重要决定应保存到项目文件或其他可靠记录中；需要回看完整讨论时再搜索历史。

大量日志、重复进度和完整文件反复进入会话，会增加压缩频率与成本。工具结果可以保留文件定位和必要片段，再按需读取细节。


## session_search：先找到会话，再看附近的证据

课件中的三种调用形态由参数决定，没有需要手动选择的 mode 字段。

| 形态 | 传入参数 | 用途 |
| --- | --- | --- |
| Discovery | query，可配 limit / sort | 按关键词寻找相关会话 |
| Scroll | session_id + around_message_id，可配 window | 围绕命中消息继续阅读 |
| Browse | 不传参数 | 查看近期会话 |

当前公开实现还支持仅传 session_id 读取指定会话等扩展形式。学习时先掌握“搜索 → 取锚点 → 阅读上下文”的主线。

结果可带会话 ID、标题、时间、命中片段、消息锚点及部分开头 / 结尾上下文。返回量受版本、数量限制与内容预算影响，不保证一次读取完整历史。

搜索范围受当前实例、Profile、权限和保留数据影响。用户已经给出文件或网址时，应先看直接来源，不能用“历史搜索没找到”证明那个来源不存在。


## 搜索与滚动：用真实返回的 ID，不编造锚点

下面用统一的“工具名 + 参数”JSON 记录表示调用思路，不是 shell 命令，也不是完整的模型服务请求格式。

~~~json
{
  "name": "session_search",
  "arguments": {
    "query": "docker deployment",
    "limit": 3,
    "sort": "newest"
  }
}
~~~

找到相关结果后，把其 session_id 和命中消息 ID 填入下一次参数：

~~~json
{
  "name": "session_search",
  "arguments": {
    "session_id": "20260610_143052_a1b2c3",
    "around_message_id": 789,
    "window": 5
  }
}
~~~

当前 schema 的 around_message_id 是整数。课件中类似 msg_789 的字符串只是示意，不能照抄为实际数据库锚点；返回字段名也要以版本为准，当前公开实现使用 match_message_id。

window=5 表示锚点两侧的窗口要求，并不保证总会有完整 11 条消息，边界、过滤与返回预算可能减少结果。Agent 应基于实际历史证据回答，区分当时的结论和今天的状态。


## FTS5 查询：关键词搜索不是语义向量检索

~~~text
docker deployment
"exact phrase"
docker OR kubernetes
python NOT java
deploy*
~~~

| 写法 | 课件中的用途 |
| --- | --- |
| 空格分隔的多个词 | 常用于同时匹配多个词 |
| 双引号短语 | 精确短语匹配 |
| OR / NOT | 组合或排除查询条件 |
| 前缀加星号 | 前缀匹配 |

角色筛选 role_filter 默认为 user,assistant 一类对话角色；调试工具时可按版本加入 tool。中文索引、标点、分词和查询转换会影响匹配，应观察实际返回而不是假定任何字符串都能原样作为 FTS 表达式执行。

课件描述 v0.15 的搜索取消了额外 LLM 摘要步骤。当前 schema 也说明返回数据库中的实际消息，不另用 LLM 做搜索摘要。**搜索本身不额外调用模型，不代表最终回答、返回文本 token 或外部服务都零费用。**

“4500 倍”“约 20ms”属于课件记录的特定测试口径，性能还取决于历史规模、索引、磁盘和返回内容，不作为本笔记的实测结果。


## Dashboard：在浏览器里查看和管理 Hermes

~~~bash
hermes dashboard
hermes dashboard --port 8080
hermes dashboard --status
hermes dashboard --stop
~~~

| 命令 | 作用 |
| --- | --- |
| dashboard | 启动浏览器管理界面 |
| --port 8080 | 使用指定端口 |
| --status | 查看相关服务进程状态 |
| --stop | 请求停止相应管理服务，具体作用范围看版本 |

默认监听地址为 127.0.0.1，端口为 9119，对应 `http://127.0.0.1:9119`。在远程服务器运行时，这个地址属于服务器自身，不会自动变成自己电脑上的本地服务。

课件还给出 dashboard --tui 来启用 Chat。当前公开解析器已经把它作为兼容旧调用的参数接受并忽略，聊天界面由新版实现提供，不能继续把它解释成所有版本都必需的开关。

本篇没有启动服务或打开实际管理页面。这里的命令展示不同操作，使用时按目的选择。


## Web 管理界面、Desktop 与后台运行，各有作用

| 入口 / 功能 | 作用 |
| --- | --- |
| Dashboard | 浏览器中的配置、会话与服务管理入口 |
| Desktop | 桌面应用中的交互入口，可连接本地或远端后端 |
| Gateway | 消息平台与远程接入相关的运行服务 |
| Web 管理功能 | 课件列出渠道、MCP、凭证、Webhook 和 Gateway 状态管理 |

界面中的可用页签依版本和配置变化，多个入口是否共享状态，仍取决于使用哪个后端和数据根目录。

课件在 Bash 中给出后台运行示例：

~~~bash
hermes dashboard > /dev/null 2>&1 &
disown
~~~

重定向与后台操作不会自动建立开机启动、进程重启和日志管理。上面的 disown 是支持它的 shell 的作业控制命令，不能原样放入 Windows PowerShell。

配置页可能管理凭证和工具权限；远程访问应使用该版本支持的认证与访问方式，而不是只改变监听地址就认为完成了安全的远端接入。


## Toolsets：选择 Agent 可以使用哪些能力

Tool 是单项能力，Toolset 按用途把能力组合起来。是否启用工具集、依赖是否可用、账号是否有权限，以及实际执行是否获准，是不同条件。

~~~bash
hermes tools
hermes tools list
hermes tools list --platform cli
hermes tools enable web --platform cli
hermes tools disable web --platform cli
~~~

enable / disable 示例表示两种相反操作，按需要选择。web 是当前公开实现中的工具集名；其他名称应从当前版本的列表中选择。

平台配置可以不同，CLI 可用的工具不一定在每个消息平台自动可用。开启工具集也不自动配置搜索服务账号、浏览器环境或工具的执行后端。

在会话内使用 /tools 查看当前可用能力，用 /verbose 观察执行过程。它们是会话命令，不应与上面的 shell 命令混在同一个命令行脚本中。


## 工具按用途分类，先记住输入输出

| 类别 | 课件中的代表能力 | 主要作用 |
| --- | --- | --- |
| 网页 | web_search、web_extract、x_search | 找资料并提取页面内容 |
| 终端和文件 | terminal、process、read_file、patch、write_file | 执行命令、管理进程、读写文件 |
| 浏览器与桌面 | browser_navigate、browser_snapshot、browser_vision、computer_use | 与实际页面或图形界面交互 |
| 媒体 | vision_analyze、image_generate、text_to_speech、video_analyze | 图像、语音和视频相关处理 |
| 编排 | 任务列表、clarify、execute_code、delegate_task | 组织步骤、澄清、运行代码与委派 |
| 记忆和召回 | memory、session_search | 保存长期事实或搜索历史 |
| 自动化和推送 | cronjob、send_message | 定时工作与对外消息 |
| 集成 | MCP、Home Assistant 等 | 接入其他系统提供的能力 |

工具名与具体参数随版本调整，媒体工具还可能需要另外的模型或服务。界面列出了一个工具，不代表任何输入、任何账号都能成功执行。

模型生成的是调用请求，实际代码由 Agent / 后端执行，再把结果交回模型；最终回答需要依据结果，不能凭请求已发出就断言执行成功。


## 搜索网页与提取正文：先定位，再阅读

搜索一般返回标题、网址与摘要；要核对具体规则或技术细节，通常还要提取并读取原始页面。

当前公开 schema 的调用参数记录如下：

~~~json
{
  "name": "web_search",
  "arguments": {
    "query": "site:hermes-agent.nousresearch.com MCP configuration",
    "limit": 5
  }
}
~~~

~~~json
{
  "name": "web_extract",
  "arguments": {
    "urls": ["https://hermes-agent.nousresearch.com/docs/user-guide/features/mcp"],
    "char_limit": 15000
  }
}
~~~

课件写 max_results、url 和 extract_mode；当前对应的是 limit、urls 列表等字段。这里按当前 schema 展示，实际调用应核对安装版本，不将这些字段任意混用。

搜索操作符受服务后端支持范围影响。页面正文也可能截断、抽取失败或过期；需检查返回状态、来源和遗漏提示。处理动态交互页面时，可以转用浏览器能力。


## 网页提取、浏览器快照与视觉分析，得到的信息不同

| 方式 | 主要得到什么 | 适用情况 |
| --- | --- | --- |
| web_extract | 页面正文等文本 | 文档阅读、资料核对 |
| browser_snapshot | 元素与可交互结构的文本表示 | 找输入框、按钮、链接等 |
| browser_vision / 视觉工具 | 截图或截图的视觉分析 | 布局、图表、纯视觉内容 |
| computer_use | 实际图形界面操作能力 | 需要点击、输入和截图的任务 |

~~~text
打开目标页面 → 检查当前结构 → 定位元素 → 执行交互 → 再检查结果
需要判断布局或图像时，再读取截图 / 使用视觉能力
~~~

快照不是完整截图，也不能保证包含页面的所有语义；视觉分析不是后台数据库查询。操作前应基于当前页面状态定位，页面变化后重新检查。

这里描述课件的能力分工，不提供未经核对的完整浏览器调用参数。具体驱动、模型、登录状态和可操作范围依配置而定。


## 文件读取与 patch：范围、唯一匹配与结果检查

下面是当前公开 schema 的参数记录。path 是目标路径，offset 从 1 开始计行，limit 控制读取的行数。

~~~json
{
  "name": "read_file",
  "arguments": {
    "path": "/home/user/project/src/main.py",
    "offset": 40,
    "limit": 80
  }
}
~~~

~~~json
{
  "name": "patch",
  "arguments": {
    "path": "/home/user/project/src/main.py",
    "old_string": "def process(data):\n    return data.strip()",
    "new_string": "def process(data: str) -> str:\n    return data.strip()",
    "replace_all": false
  }
}
~~~

路径是示例，需替换为工具后端中的实际文件。课件使用 file_path，而当前这两个工具使用 path。

默认替换需要能唯一定位目标，必要时包含周围代码。当前 patch 还有模糊匹配等能力，所以不能把它永远解释成严格逐字节匹配。replace_all 表示有意替换所有匹配项，不能拿它掩盖目标不明确的问题。

检查返回 diff，再核对文件与必要的运行行为。操作“成功写入”和改动“正确解决问题”是两个结论。


## 任务列表：计划、进度和完成证据

课件用 todo(action="create" / "update") 演示规划。当前公开实现的工具名是 todo_list，使用 todos 与 merge；下面展示当前参数形状。

~~~json
{
  "name": "todo_list",
  "arguments": {
    "todos": [
      {"id": "search", "content": "查找官方 MCP 文档", "status": "in_progress"},
      {"id": "read", "content": "读取并核对配置要求", "status": "pending"},
      {"id": "write", "content": "整理配置说明并检查格式", "status": "pending"}
    ],
    "merge": false
  }
}
~~~

~~~json
{
  "name": "todo_list",
  "arguments": {
    "todos": [
      {"id": "search", "content": "查找官方 MCP 文档", "status": "completed"},
      {"id": "read", "content": "读取并核对配置要求", "status": "in_progress"}
    ],
    "merge": true
  }
}
~~~

merge=false 用新的列表替换计划；merge=true 按 ID 更新已有项或添加新项。不要在只更新两项时误用整体替换，导致其他待做工作消失。

计划帮助防止漏项，但不是完成证明。标记 completed 前，仍需确认相应资料已读取、改动已保存或检查已通过。


## 终端后端：命令在哪台机器、哪个环境执行？

| 后端 | 课件中的用途 | 需要准备 |
| --- | --- | --- |
| local | 在 Agent 所在本机执行 | 本机环境与权限 |
| docker | 在容器中执行 | Docker 服务、镜像和所需挂载 |
| ssh | 在远程服务器执行 | 连接配置、认证及远端权限 |
| singularity | HPC / Apptainer 类容器环境 | 集群运行环境与镜像 |
| modal | 云端执行资源 | 相应账号、凭证与镜像 |
| daytona | 云端开发工作区 | 相应工作区及服务配置 |
| vercel_sandbox | 云端沙箱环境 | 服务支持、凭证与资源限制 |

~~~bash
hermes config set terminal.backend docker
hermes config set terminal.backend local
~~~

两条命令表示切换到不同后端，按需要选择。安装版本支持哪些后端，以实际配置和文档为准。

SSH 是远程执行方式，不因名称就自动具备沙箱隔离；容器挂载宿主机目录、继承凭证或暴露服务时，也会影响实际边界。首先确认命令所在环境，以及哪些文件和服务能被它访问。

终端后端只控制对应工具的运行环境，不保证本地 MCP Server、浏览器或其他工具也自动迁入同一个容器。


## Docker 配置：工作目录不等于自动挂载项目

下面保留课件镜像和少量关键配置，帮助理解各项作用，不是当前版本的完整默认文件。

~~~yaml
terminal:
  backend: docker
  cwd: /workspace
  timeout: 180
  docker_image: nikolaik/python-nodejs:python3.11-nodejs20
  docker_mount_cwd_to_workspace: false
  docker_volumes: []
  persistent_shell: true
~~~

| 配置 | 含义 |
| --- | --- |
| cwd | 后端中的工作目录 |
| timeout | 单次命令的超时设置，单位与行为核对当前版本 |
| docker_image | 容器镜像，版本需匹配任务依赖 |
| docker_mount_cwd_to_workspace | 是否采用该实现提供的项目目录挂载行为 |
| docker_volumes | 自定义挂载列表，读写范围需明确 |
| persistent_shell | 是否保持 shell 状态，具体持久化行为看实现 |

cwd=/workspace 不会凭空把 Windows 或 Linux 的项目文件放进去。没有项目挂载或文件复制时，容器未必能看到目标代码；可写挂载的修改可能直接反映到宿主机。

课件镜像中的 Python / Node 版本不代表自己安装的 Hermes 运行时版本。选择镜像要检查实际任务依赖，不能因课件给出一个 tag 就保证今后所有依赖兼容。


## Docker 镜像源配置，属于 Docker 自身

课件在终端配置旁插入了 registry-mirrors 片段。它应按 Docker 的配置格式放到相应 daemon 设置中，不是 Hermes config.yaml 的一个字段。

~~~json
{
  "registry-mirrors": []
}
~~~

这里仅演示有效 JSON 的完整形状。需要镜像源时，列表填入自己验证可用的地址，并与现有 Docker 配置合并。课件列出的第三方镜像地址不能视为长期可用保证。

拉取课件镜像的命令示例为：

~~~bash
docker pull nikolaik/python-nodejs:python3.11-nodejs20
~~~

镜像拉取成功只说明镜像取得，不代表容器已经启动、项目已经挂载或工具任务已经通过。本笔记没有修改 Docker 设置或拉取镜像。


## 工具并行、审批和写入检查，分别保护什么？

独立调用可以并行，例如读取不同文件或查询不同来源；依赖前一步结果的调用则应保持顺序。同一个文件的并发编辑、先写后读的业务流程，不应随意并行。

课件提到 ThreadPoolExecutor 与最多 8 个 worker，属于该版本实现说明。MCP 工具的并行策略还可能需要服务器级别配置，不能推断所有工具都会以同样并发度运行。

| 机制 | 解决的问题 | 无法单独证明 |
| --- | --- | --- |
| 并行调度 | 减少独立操作的等待 | 操作之间完全没有冲突 |
| Smart Approvals | 根据策略处理工具执行确认 | 某命令在任何目录和参数下都安全 |
| /yolo | 改变确认流程 | 获得了额外系统权限或结果一定正确 |
| Post-Write Linting | 写入后检查部分文件格式 | 程序语义、依赖和业务逻辑正确 |

课件介绍 patch / write_file 后对 Python、JSON、YAML、TOML 等的格式检查。解析通过仍可能有名称错误、错误规则或运行失败，重要改动要检查实际 diff 并完成适合该改动的验证。

减少审批提示与减少执行日志也是两件事。/verbose 的 off、new、all、verbose 控制展示程度，不等于工具被禁用；调试时应保留足够信息定位失败步骤。


## MCP：统一工具接入协议，模型仍需会使用工具

MCP（Model Context Protocol）连接客户端和工具服务器，让服务器提供的能力被客户端发现并调用。

~~~text
Hermes 作为 MCP 客户端
  → 启动本地 stdio Server，或连接远程 HTTP Server
  → 发现工具与参数 schema
  → 注册可用工具
  → 模型发起调用 → 服务器执行 → 返回结果
~~~

协议统一接入方式，不自动补齐账号权限、依赖、网络、业务授权或模型的工具调用能力。工具 schema 告诉模型能传什么参数，实际服务仍负责执行和访问控制。

本章区分两条方向：Hermes 接入外部 MCP，和其他客户端把 Hermes 当成 MCP Server。两者的配置文件位置、能力范围与进程管理都不同。


## stdio MCP：在本地启动一个工具进程

下面示例把指定目录提供给文件系统 MCP。C:/work/demo 是占位项目目录，需要先准备实际目录；npx 所属环境应能取得并运行指定包。

~~~yaml
mcp_servers:
  project-fs:
    command: npx
    args:
      - "-y"
      - "@modelcontextprotocol/server-filesystem"
      - "C:/work/demo"
    enabled: true
    connect_timeout: 30
    timeout: 60
~~~

| 配置 | 用途 |
| --- | --- |
| command + args | 启动服务器进程的命令和参数 |
| env | 传给该进程的环境配置，需要时设置 |
| enabled | 是否启用服务器 |
| connect_timeout | 建立连接和发现工具的超时 |
| timeout | 后续工具调用的超时 |

stdio 通过标准输入 / 输出交换协议消息，进程的诊断日志应遵循服务器要求，不能用无关输出破坏协议流。

Linux 可将目录改为 /home/user/my-project，Windows 使用实际 Windows 路径。这个 Server 的运行位置和允许访问目录由启动配置决定，不由 terminal.backend 自动控制。


## HTTP MCP：连接已经运行的服务

~~~yaml
mcp_servers:
  company-api:
    url: "https://mcp.internal.example.com/mcp"
    headers:
      Authorization: "Bearer ${COMPANY_MCP_TOKEN}"
    enabled: true
    connect_timeout: 30
    timeout: 60
~~~

当前官方文档支持在 headers、env 等位置解析 ${VAR} 引用。把实际令牌放到对应实例的环境配置，而不是直接写进公开 YAML：

~~~dotenv
COMPANY_MCP_TOKEN=your_token_here
~~~

网址和令牌是占位值，示例不会连接真实企业服务。配置 URL 不会自动创建服务，也不保证路由、证书、认证和工具列表都可用。

需要 OAuth 的 HTTP Server，可按该版本说明使用 auth: oauth，并完成交互授权。OAuth 与 Bearer 头是不同认证路线，不应为了“多写点配置”就不加区分地同时堆上。


## 工具筛选与重载：配置了 Server，不代表所有能力都已启用

可以在前面的 project-fs 节点下追加工具筛选。以下名称是常见文件系统 Server 的示意，实际名称必须来自该 Server 的工具发现结果。

~~~yaml
tools:
  include:
    - list_directory
    - read_text_file
~~~

tools.include 选择明确的一组工具，tools.exclude 排除指定工具。过滤的是在 Hermes 中注册的能力，不替代服务端账号权限或其他客户端的访问控制。

~~~bash
hermes mcp list
hermes mcp test project-fs
hermes mcp configure project-fs
hermes mcp remove project-fs
~~~

list 查看配置，test 检查连接，configure 管理工具选择，remove 移除配置；它们是不同操作，移除不等同于卸载所有依赖包。

修改后在会话中使用：

~~~text
/reload-mcp
~~~

重载会刷新相应连接和工具列表。服务器支持动态工具变更通知时，还可能自动刷新；具体可用性与通知机制依版本，不能只改 YAML 就假定所有已打开会话立即采用新工具。


## Hermes 作为 MCP Server：看实际暴露的接口

客户端接入配置与 Hermes 自己的 mcp_servers 不同。下面是采用 mcpServers 格式的客户端配置示意：

~~~json
{
  "mcpServers": {
    "hermes-agent": {
      "command": "hermes",
      "args": ["mcp", "serve"]
    }
  }
}
~~~

MCP 客户端负责启动和管理这个 stdio 进程，其运行环境需能找到 hermes；图形客户端的 PATH 可能与交互终端不同，必要时使用实际入口的绝对路径。

课件把该模式描述成“调用 Hermes 的所有已启用工具”。当前官方文档公开的主要是消息与会话桥接接口，例如 conversations_list、messages_read、messages_send 等。**不能将 MCP Server Mode 等同于把所有内置工具原样无限透传。**

发送消息还需要对应平台适配器与运行连接，涉及真实外部动作。这里只解释接口，不配置客户端或发送消息。

Claude 系列客户端、Cursor 和 VS Code 的 MCP 配置位置和 schema 可能不同，不能保证每个产品都使用同一个 mcpServers 块。


## MCP Catalog：目录帮助发现，仍要检查安装结果

~~~bash
hermes mcp catalog
~~~

课件把 Catalog 介绍为精选服务器的发现入口。当前实现区分目录列表与交互 picker，界面行为要按版本看帮助。

也可以参考 [魔搭 MCP 广场](https://www.modelscope.cn/mcp) 等目录了解可用服务。目录条目可能提供本地包、远程地址或配置模板，它们并不是同一种安装方式。

接入时确认来源、启动命令 / URL、凭证、可用工具与实际访问范围，再通过 test 和一个范围明确的读取调用检查。目录收录不代表自己环境的依赖、账号和网络问题已经解决。


## 上下文文件：项目规则与实例风格分开保存

| 文件 | 主要用途 | 课件中的发现方式 |
| --- | --- | --- |
| .hermes.md / HERMES.md | Hermes 专用项目上下文 | 从当前目录向 Git 根目录寻找 |
| AGENTS.md | 项目架构、约定和操作说明 | 启动或访问子目录时发现 |
| CLAUDE.md | 兼容的项目上下文 | 按支持的规则发现 |
| .cursorrules / .cursor/rules/*.mdc | Cursor 风格项目规则 | 按支持的规则加载 |
| SOUL.md | 当前实例的身份与沟通风格 | 从 HERMES_HOME 独立加载 |

课件给出的启动类型优先级为：

~~~text
.hermes.md / HERMES.md
  → AGENTS.md
  → CLAUDE.md
  → .cursorrules / .cursor/rules/*.mdc
~~~

优先级是选择项目上下文类型的规则，不代表把所有文件无条件累加。SOUL.md 独立加载，不参与上述类型竞争。

当前实现对 AGENTS.md 还可能沿 Git 根目录到当前目录组织规则链。具体文件名、覆盖关系与路径边界，按安装版本核对；不要认为根目录和子目录的规则永远具有同样作用。


## 启动加载与渐进加载：在不同时间进入上下文

| 阶段 | 课件描述 | 内容进入哪里 |
| --- | --- | --- |
| 启动加载 | 会话开始时选择项目上下文、读取和扫描 | 系统提示词中的 Project Context |
| 渐进加载 | 工具访问文件后，从路径推断相关目录并发现规则 | 相应工具结果附加的上下文 |

课件的渐进流程最多向上检查 5 层，每层按 AGENTS.md、CLAUDE.md、.cursorrules 等规则选择匹配项。它不是运行中持续扫描整个硬盘的全部规则文件。

### 课件的截断预算

启动内容以 20000 字符为示例上限，保留前 70% 和后 20%，即约 14000 + 4000 字符，中间加入截断提示；渐进文件以只保留前 8000 字符为示例。

这些是字符预算，不是 token 数。截断也意味着某些规则没有完整进入模型；出现提示后，应该按需读取完整文件。

当前公开源码已可以根据模型窗口调整部分启动预算，20000 不再应被说成所有版本和所有模型的固定最大值。重要且简短的规则放到前面，较长说明放到可按需读取的文档。


## 一个项目说明示例：写出可以核对的约定

下面是 AGENTS.md 的内容示意，全部作为笔记里的 Markdown 代码段保存，没有在项目中创建规则文件。

~~~markdown
# 项目约定

## 项目结构
- API 代码位于 src/api，业务逻辑位于 src/services。
- 运行入口与依赖说明以 README.md 为准。

## 修改方式
- 先阅读目标模块已有实现和相关说明。
- 保留接口行为，修改时说明影响范围。
- 新增配置项时同步更新配置文档。

## 验证方式
- 按 README.md 中的验证命令检查受影响功能。
- 检查实际 diff，并说明没有完成的验证。
~~~

说明应当与项目真实结构一致，避免过时路径、无法执行的命令或彼此矛盾的规则。仓库约定适合跟随项目版本管理；用户长期风格与项目临时任务不要全部挤在一个文件里。

规则文件提供上下文，不能代替工具权限、业务校验或操作系统访问控制。


## SOUL.md 与 /personality：长期风格与会话选择

SOUL.md 用于当前 Hermes 实例的身份和沟通风格，位于其数据根目录，课件将它放在系统提示词开头部分。

~~~markdown
# 沟通风格

- 使用中文解释技术问题，先说明结果，再说明依据。
- 信息不确定时说明哪些已核实、哪些仍需检查。
- 给出清楚的操作步骤和示例，避免重复表达。
~~~

这是文件内容示例，不是让本次笔记任务新建或修改 SOUL.md。

会话内可以选择预设：

~~~text
/personality concise
/personality technical
/personality teacher
~~~

课件还列出 creative、helpful、pirate 等风格，具体列表以版本为准。/personality 添加对应的交互风格提示，不等于改写磁盘上的 SOUL.md。

项目目录、测试命令和仓库规则放在项目上下文；实例风格放在 SOUL.md；用户稳定偏好可按需要进入 USER.md。尽量减少重复且互相冲突的描述。


## 上下文扫描：识别可疑内容，不能承诺完全防护

课件介绍对上下文文件的注入模式和不可见字符扫描，例如要求覆盖先前指令、隐瞒行为、泄露凭证，或利用隐藏 HTML 和双向文本字符干扰阅读。

~~~text
读取文件 → 扫描内容 → 接受并组织上下文
                    → 命中风险时阻止加载，并显示 BLOCKED 提示
~~~

检测既可能漏掉变形攻击，也可能误判正常说明。看到阻止提示时，应检查具体文件与来源，而不是简单关闭检查或假定文件内容已经进入上下文。

这类保护与工具输出标记、控制文件保护等机制可以一起使用，但不能保证所有外部文档、网页或记忆都绝对可信。理解资料和执行其中的指令仍是不同操作。

本章只解释课件机制，没有修改扫描策略、创建测试攻击文件或改变执行确认设置。


## 持久记忆：把少量稳定事实带到下一次会话

课件的内置记忆分为两个文件，默认位于数据根目录的 memories 中。

| 文件 | 用途 | 课件默认字符预算 |
| --- | --- | --- |
| MEMORY.md | 稳定环境事实、全局约定等笔记 | 2200 字符 |
| USER.md | 用户信息、沟通偏好和习惯 | 1375 字符 |

~~~text
Hermes 数据目录/
└─ memories/
   ├─ MEMORY.md
   └─ USER.md
~~~

Windows 原生通常在 `%LOCALAPPDATA%\hermes\memories\`；Linux / macOS / WSL 通常在 `~/.hermes/memories/`。HERMES_HOME 与 Profile 可以改变实际位置。

这些文件不是大容量知识库。2200 + 1375 字符也不是固定等于 1300 tokens，中文、英文和所用 tokenizer 都会影响实际 token 数。

条目可能通过 § 分隔，并显示当前容量占用。优先保存简短、明确、长期有用且尚未被其他规则文件覆盖的事实。


## 记忆快照：磁盘已更新，当前提示词不一定已更新

课件和官方记忆文档描述的模式是：在会话开始时捕获记忆快照，注入系统提示词；当前会话中的 memory 写入立即持久化，但既有系统提示快照保持稳定。

~~~text
会话 A 开始 → 读取当时的 MEMORY.md / USER.md → 固定提示词快照
会话 A 中写入新事实 → 磁盘更新，工具结果可显示最新内容
会话 B 之后开始 → 读取更新后的文件 → 使用新的快照
~~~

固定前缀有利于模型提示缓存。已经在运行的另一个会话，也不能因为文件变了就被假定立即取得新快照。

当前会话仍能从用户消息和工具结果知道刚更新的事实，这与“系统提示词中的记忆快照刷新”是两回事。

自然任务边界使用 /new 后，再确认新会话读取了正确实例的记忆。若下次会话仍没记住，检查写入结果、容量、目标文件、功能开关与数据根目录，而不是只重复说“请记住”。


## 内置记忆配置：开关与预算分别控制

~~~yaml
memory:
  memory_enabled: true
  user_profile_enabled: true
  memory_char_limit: 2200
  user_char_limit: 1375
~~~

| 配置 | 课件中的作用 |
| --- | --- |
| memory_enabled | 内置记忆笔记的开关 |
| user_profile_enabled | 用户画像的开关 |
| memory_char_limit | MEMORY.md 的字符预算 |
| user_char_limit | USER.md 的字符预算 |

配置要合并到已有 memory 节点，不能把整份 config.yaml 替换成这四项。当前版本还可能根据开关调整可写目标，关闭一个目标不代表同时关闭另一个目标或历史会话存储。

增大字符预算会增加提示内容，不保证记忆质量更好。更重要的是删除过时事实、合并重复条目，并将局部项目规则和较长流程放回合适文件。


## memory 工具：新增、替换、删除都有明确目标

课件和当前公开 schema 的基本动作是 add、replace、remove；target 区分 memory 与 user，没有用于普通读取的 read 动作。

下面是记录用户稳定偏好的参数示意：

~~~json
{
  "name": "memory",
  "arguments": {
    "action": "add",
    "target": "user",
    "content": "用户希望技术笔记使用中文说明，代码放入带语言标识的 Markdown 代码段。"
  }
}
~~~

| 动作 | 主要参数 | 含义 |
| --- | --- | --- |
| add | target、content | 添加新条目 |
| replace | target、old_text、content | 定位已有条目，再用完整新条目替换 |
| remove | target、old_text | 定位并删除已有条目 |

读当前记忆主要通过启动时注入的快照及工具结果，不应编造 memory(action="read")。需要人工核对文件时也要先确认实例与实际路径。

示例没有真的把这条偏好写入 Hermes。一次调用是否成功，要看明确的写入结果，而不是只看模型在回答里说“我记住了”。


## old_text 定位条目，replace 替换的是整条记忆

假设已有条目为“用户在 VS Code 和终端都使用深色模式。”，现在纠正为编辑器浅色、终端深色，可以用能唯一匹配的片段定位。

~~~json
{
  "name": "memory",
  "arguments": {
    "action": "replace",
    "target": "user",
    "old_text": "VS Code 和终端都使用深色模式",
    "content": "用户在 VS Code 中使用浅色模式，在终端中使用深色模式。"
  }
}
~~~

old_text 通常做短唯一子串匹配，需要刚好定位一个条目。没有匹配或匹配多条时，应读取实际结果并改用更明确的文本，不猜测编号或不断添加重复事实。

**old_text 是定位依据，content 是整条新记忆。**它不是把条目中一个短词替换掉的文本 patch；想保留的其他事实需要一并写入新的完整条目。

删除过时条目的参数形状为：

~~~json
{
  "name": "memory",
  "arguments": {
    "action": "remove",
    "target": "user",
    "old_text": "VS Code 中使用浅色模式"
  }
}
~~~

这也是针对示例已有内容的操作说明，不是本次任务实际执行的删除。


## 容量满了：整理事实，不把记忆变成日志仓库

| 信息 | 更合适的位置 |
| --- | --- |
| 用户的稳定沟通偏好、身份事实 | USER.md |
| 跨任务经常需要的稳定环境事实 | MEMORY.md |
| 当前项目架构、路径与验证命令 | 项目上下文文件 |
| 可复用的操作流程与经验 | 对应 Skill，第 03 篇展开 |
| 某次讨论、临时决策和执行细节 | 会话历史，需要时 session_search |
| 原始代码、日志、表格或大量资料 | 对应文件与知识存储 |
| 真实密钥与访问令牌 | 专门的认证配置 |

事实用陈述句记录，例如“用户偏好简洁的技术解释”，不要在记忆里塞进要求覆盖其他规则的命令。

预算不足时先删除过时项、合并重复项、缩短内容或转移到合适文档。替换也受容量约束，不是只有新增才会失败。

当前公开 schema 还支持 operations 批量更新，按最终结果预算处理多项变更；这属于当前版本扩展，复现课件版本时先确认支持范围。自动扫描与写入保护有助于减少污染，但仍需检查错误事实、冲突和不该保存的内容。


## session_search 与 memory：大历史按需找，小事实常驻

| 对比 | 持久记忆 | session_search |
| --- | --- | --- |
| 内容 | 少量高价值稳定事实 | 被保存的历史会话消息 |
| 进入上下文 | 会话开始时的记忆快照 | 查询后返回的相关内容 |
| 管理 | 主动新增、替换、删除 | 会话保存、索引、恢复和清理 |
| 适合问题 | 用户通常希望什么、稳定环境是什么 | 上次讨论怎样解决、当时为何决定 |
| 预算 | 有严格字符上限 | 返回数量、窗口与内容预算受限 |
| 费用关系 | 占用提示词，缓存与计费依模型服务 | 搜索可不额外调用 LLM，但结果和最终回答仍占上下文 |

已保存会话不是无限容量保证：数据可能被清理、未写入当前实例或超出可访问范围。历史也不自动证明现在状态没有改变。

例如“用户希望学习笔记代码放在 Markdown 代码段中”是稳定偏好；“上次 Docker 端口冲突最终改了哪个配置”则应找对应会话或当前文件，不必把整段排查日志放进常驻记忆。


## 外部记忆 Provider：在内置记忆之外增加检索能力

课件介绍外部插件与内置记忆叠加工作，同一配置通常选择一个外部提供商。

~~~bash
hermes memory setup
hermes memory status
hermes memory off
~~~

setup 配置提供商，status 查看状态，off 关闭相应外部记忆能力。它们不应直接被解释为删除 MEMORY.md、USER.md 或全部历史。

| 课件列出的 Provider | 课件强调的方向 |
| --- | --- |
| honcho | 用户建模、跨会话理解 |
| mem0 | 事实抽取、语义搜索与去重 |
| openviking | 分层、文件系统式记忆组织 |
| byterover | CLI 驱动的层级知识管理 |
| hindsight | 实体关系、知识图谱和关联检索 |
| holographic | 全文搜索、评分与冲突处理 |
| retaindb | 混合搜索、分类记忆和增量处理 |
| supermemory | 语义长期记忆与用户画像 |

这些是课件的选型方向，不是已经在本机验证的能力清单或固定优劣排名。插件数量、后端部署、账号、模型抽取和数据传输都要按安装版本和所选服务核对。

入门先确认内置记忆与历史检索是否满足需要，再依据记忆规模、更新频率、检索质量、部署位置和费用选择外部方案。


## 串起一次项目任务：各能力在什么时候参与？

| 阶段 | 用到的能力 | 应该检查的结果 |
| --- | --- | --- |
| 进入项目 | 项目上下文 + SOUL.md + 记忆快照 | 路径、规则与实例正确 |
| 用户提及上次讨论 | session_search | 找到真实讨论和相关结论 |
| 收集资料 | 文件、网页或 MCP 工具 | 来源、返回状态与内容完整性 |
| 修改与执行 | patch、终端后端等 | 实际 diff、运行环境与必要检查 |
| 多步骤推进 | 任务列表与过程展示 | 各步骤完成证据，避免遗漏 |
| 任务结束 | 总结、项目文件、适当记忆更新 | 结果保存，长期事实与临时细节分开 |
| 新任务开始 | /new 或指定恢复 | 会话边界与需要的经验正确载入 |

项目规则说明怎样处理项目，工具负责实际操作，历史用于回溯，记忆用于保留稳定事实。它们相互配合，不是任选一个就能代替其余机制。

涉及外部消息、付款、部署或真实数据修改时，还需要相应的任务授权与业务校验。课件中的工具名称只是能力介绍，没有授权本次笔记任务执行这些动作。


## 常见问题：先确定失败发生在哪一层

| 问题 | 排查重点 |
| --- | --- |
| 继续会话恢复了错误任务 | ID、标题、终端选择规则、工作目录与 Profile |
| /undo 后文件仍改变 | 交互记录与外部效果的区别，检查实际 diff / 检查点 |
| 历史搜索没有命中 | 关键词、语言索引、角色、实例和历史是否仍保存 |
| 压缩后遗漏重要细节 | 摘要质量、保护消息、项目记录及完整历史 |
| Dashboard 地址打开不了 | 服务在哪台机器、实际端口、进程和网络 |
| 启用工具后仍不可用 | 当前会话刷新、依赖、认证和后端 |
| Docker 中看不到项目 | cwd 与真实挂载 / 文件复制是否一致 |
| MCP 配置成功但调用失败 | 进程 / URL、认证、发现的工具、过滤和调用 schema |
| 项目规则未生效 | 文件类型优先级、路径范围、截断与阻止加载提示 |
| 说记住了，下次却没有 | 工具写入结果、容量、开关、正确实例和新快照 |
| 替换记忆丢失其他信息 | content 是否包含完整的新条目 |

先收集明确的错误与返回结果，再调整对应层。反复重跑安装或把所有配置全部删除，通常不会帮助确认是哪一步出了问题。


## 本章回顾与复习问答

本章把 Agent 的持续工作拆成六部分：会话保存任务，Dashboard 提供管理入口，工具执行操作，MCP 接入外部服务，上下文文件描述规则，持久记忆保留少量长期事实。

| 问题 | 回答要点 |
| --- | --- |
| 历史保存是否代表模型每次看到全部历史？ | 不代表，实际提示受窗口、压缩和按需检索影响 |
| /compress 是否会删除历史？ | 它主要整理当前上下文，不等同于历史删除 |
| /undo 是否能保证回滚工具操作？ | 不能据交互撤销就作此判断，外部结果需另查 |
| session_search 为什么不等于完全免费？ | 搜索可不另调 LLM，但返回文本与最终回答仍有模型成本 |
| terminal.backend=docker 是否会自动隔离所有工具？ | 不会，还要看其他工具进程、挂载与凭证 |
| MCP 连接成功是否表示工具都可用？ | 不一定，发现、过滤、schema、账号和执行结果要分别检查 |
| MCP Server Mode 是否就是全部工具透传？ | 看实际暴露的接口，当前公开实现主要是消息 / 会话桥接 |
| 项目文件与 SOUL.md 的职责不同在哪里？ | 项目文件描述仓库规则，SOUL.md 描述实例身份与沟通风格 |
| 记忆写入为何可能要下个会话才更新提示词？ | 会话开始时捕获固定快照，磁盘与当前提示快照不同 |
| memory replace 是短字符串替换吗？ | old_text 定位条目，content 覆盖整条记忆 |

下一章学习 Skills、Curator、Hooks、Plugins 和 Cron，进一步把经验变成可复用流程。


## 资料来源与版本说明

主资料：[原始课件包](<D:/baidudisk/BaiduNetdiskDownload/1、课件(1)/1.课件.rar>) 中的《02-能力篇.md》，六个模块为会话、Dashboard、Toolsets、MCP、上下文文件和持久记忆，课程标注基准版本为 v0.16.0。

- [会话管理](https://hermes-agent.nousresearch.com/docs/user-guide/sessions)
- [Dashboard](https://hermes-agent.nousresearch.com/docs/user-guide/features/web-dashboard)
- [工具说明](https://hermes-agent.nousresearch.com/docs/user-guide/features/tools)
- [MCP](https://hermes-agent.nousresearch.com/docs/user-guide/features/mcp)
- [上下文文件](https://hermes-agent.nousresearch.com/docs/user-guide/features/context-files)
- [人格与风格](https://hermes-agent.nousresearch.com/docs/user-guide/features/personality)
- [内置记忆](https://hermes-agent.nousresearch.com/docs/user-guide/features/memory)
- [外部记忆](https://hermes-agent.nousresearch.com/docs/user-guide/features/memory-providers)
- [公开源码](https://github.com/NousResearch/hermes-agent)

整理时对照了当前公开的 session_search、网页、文件、任务列表、memory 工具 schema，CLI 参数解析器，以及上下文与压缩实现。正文明确区分课件基准说明和当前接口变化，不把公开源码核对当作本机运行测试。

完成的验证范围是 notebook 结构、Markdown 围栏、配置和参数示例的静态检查与渲染。没有实际调用 Hermes 工具、启动服务、执行容器、修改记忆或清理会话，也没有测得课件宣传的性能数据。
